# 04 — Ensemble (TF-IDF + XGBoost + DeBERTa)

Combines the two ML layers into a final decision.

Two combination strategies are computed:
1. **Weighted vote** — transparent, no extra training
2. **Logistic regression meta-model** trained on OOF predictions — the learned combination

Outputs: `artifacts/ensemble_oof.csv`, `artifacts/ensemble_test.csv` (consumed by `06_evaluation.ipynb`) and `artifacts/meta_logistic.joblib`.

In [1]:
!pip -q install -U "scikit-learn>=1.9,<2" "joblib>=1.4,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.8 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from google.colab import drive
drive.mount('/content/drive')

from sklearn.linear_model import LogisticRegression

PROJECT_DIR = Path('/content/drive/MyDrive/softcom-prompt-injection')
PROC_DIR = PROJECT_DIR / 'data' / 'processed'
ART_DIR = PROJECT_DIR / 'artifacts'

Mounted at /content/drive


### Load OOF + test predictions from notebook 02 (TF-IDF+XGBoost) and notebook 03 (DeBERTa)

In [3]:
dev = pd.read_csv(PROC_DIR / 'dev.csv')
test = pd.read_csv(PROC_DIR / 'test.csv')

tfidf_oof = pd.read_csv(ART_DIR / 'tfidf_xgb_oof.csv')
tfidf_test = pd.read_csv(ART_DIR / 'tfidf_xgb_test.csv')
deberta_oof = pd.read_csv(ART_DIR / 'deberta_oof.csv')
deberta_test = pd.read_csv(ART_DIR / 'deberta_test.csv')

# Merge by row_id, never rely on file order
oof = dev[['row_id', 'label', 'text']].merge(tfidf_oof, on='row_id').merge(deberta_oof, on='row_id')
tst = test[['row_id', 'label', 'text']].merge(tfidf_test, on='row_id').merge(deberta_test, on='row_id')

assert len(oof) == len(dev)
assert len(tst) == len(test)
assert set(oof['row_id']) == set(dev['row_id'])
assert set(tst['row_id']) == set(test['row_id'])

print('OOF shape:', oof.shape)
print('Test shape:', tst.shape)

OOF shape: (824, 5)
Test shape: (207, 5)


### Weighted-vote reference
Re-weighted for two layers (previously 0.50 / 0.35 / 0.15 across three). DeBERTa keeps the larger share since it consistently outperforms the lexical model on semantic/paraphrased attacks.

In [4]:
META_FEATURES = ['deberta_prob', 'tfidf_xgb_prob']
WEIGHTS = np.array([0.60, 0.40], dtype=float)

X_oof = oof[META_FEATURES].to_numpy(dtype=np.float64)
y_oof = oof['label'].to_numpy(dtype=int)
X_test = tst[META_FEATURES].to_numpy(dtype=np.float64)
y_test = tst['label'].to_numpy(dtype=int)

weighted_oof_prob = np.average(X_oof, axis=1, weights=WEIGHTS)
weighted_test_prob = np.average(X_test, axis=1, weights=WEIGHTS)

### Logistic meta-model, trained on OOF predictions only (never on in-sample predictions)

In [5]:
meta_model = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
meta_model.fit(X_oof, y_oof)

oof_meta_prob = meta_model.predict_proba(X_oof)[:, 1]
test_meta_prob = meta_model.predict_proba(X_test)[:, 1]

print('Meta coefficients:')
for feature, coef in zip(META_FEATURES, meta_model.coef_[0]):
    print(f'  {feature:20s}: {coef:+.6f}')
print('Meta intercept:', float(meta_model.intercept_[0]))

Meta coefficients:
  deberta_prob        : +8.863956
  tfidf_xgb_prob      : +2.944629
Meta intercept: -7.2330065133985135


### Save combined prediction tables (consumed by `06_evaluation.ipynb`)

In [6]:
oof_out = oof[['row_id', 'label', 'deberta_prob', 'tfidf_xgb_prob']].copy()
oof_out['weighted_vote_prob'] = weighted_oof_prob
oof_out['meta_prob'] = oof_meta_prob
oof_out.to_csv(ART_DIR / 'ensemble_oof.csv', index=False)

test_out = tst[['row_id', 'text', 'label', 'deberta_prob', 'tfidf_xgb_prob']].copy()
test_out['weighted_vote_prob'] = weighted_test_prob
test_out['meta_prob'] = test_meta_prob
test_out.to_csv(ART_DIR / 'ensemble_test.csv', index=False)

print('Saved:', ART_DIR / 'ensemble_oof.csv')
print('Saved:', ART_DIR / 'ensemble_test.csv')

Saved: /content/drive/MyDrive/softcom-prompt-injection/artifacts/ensemble_oof.csv
Saved: /content/drive/MyDrive/softcom-prompt-injection/artifacts/ensemble_test.csv


### Save the meta-model + metadata
The metadata file records what the meta-model expects as input — useful later whether you build the app yourself with joblib + transformers directly, or come back for help wiring it up.

In [7]:
joblib.dump(meta_model, ART_DIR / 'meta_logistic.joblib')

THRESHOLD = 0.50
metadata = {
    'model_labels': {'0': 'BENIGN', '1': 'ATTACK'},
    'threshold': THRESHOLD,
    'meta_features': META_FEATURES,
    'deberta_model_name': 'microsoft/deberta-v3-base',
    'max_length': 256,
    'weights_reference': {'deberta': float(WEIGHTS[0]), 'tfidf_xgboost': float(WEIGHTS[1])},
}
(ART_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved metadata:', metadata)
print('Saved model:', ART_DIR / 'meta_logistic.joblib')

Saved metadata: {'model_labels': {'0': 'BENIGN', '1': 'ATTACK'}, 'threshold': 0.5, 'meta_features': ['deberta_prob', 'tfidf_xgb_prob'], 'deberta_model_name': 'microsoft/deberta-v3-base', 'max_length': 256, 'weights_reference': {'deberta': 0.6, 'tfidf_xgboost': 0.4}}
Saved model: /content/drive/MyDrive/softcom-prompt-injection/artifacts/meta_logistic.joblib
